# Day 044 | Support Vector Machines (SVM)

## 100 Days of ML for Geotechnical Engineering
**Phase:** Classical Machine Learning

**Author:** Ripon Chandra Malo  
**Date:** YYYY-MM-DD  

---

## What We Will Learn Today

| Concept | What It Means | Geotechnical Example |
|---------|--------------|---------------------|
| SVM (Support Vector Machine) | Finds the best boundary between classes | Separating liquefiable vs non-liquefiable soils |
| Hyperplane | Decision boundary in feature space | The line/surface dividing safe vs unsafe soil |
| Margin | Distance from boundary to nearest points | How confident the separation is |
| Support Vectors | Data points closest to the boundary | Critical soil samples that define the limit |
| Kernel Trick | Maps data to higher dimensions | Handling non-linear liquefaction boundaries |
| C parameter | Controls margin vs misclassification tradeoff | Strict vs tolerant classification |
| Gamma parameter | Controls influence radius of each point | Local vs global pattern sensitivity |
| Decision Boundary | Visual boundary between classes | 2D map of liquefaction zones |

---

## Our Geotechnical Problem

**Soil Liquefaction Classification**

During earthquakes, saturated loose sands can lose their strength and behave like a liquid — this is **liquefaction**. Engineers need to predict whether a soil layer will liquefy based on field test data.

Key factors:
- **N-value (SPT blow count):** Higher N = denser soil = less likely to liquefy
- **Depth:** Deeper soils have more confining pressure
- **Fines content:** More fines can reduce liquefaction potential
- **Water table depth:** Saturation is required for liquefaction

We will build an SVM classifier that draws the optimal boundary between liquefiable and non-liquefiable soils.

---

## Part 1: What is an SVM?

A **Support Vector Machine** finds the hyperplane that **maximizes the margin** between two classes.

Think of it this way:
- You have soil samples plotted by N-value (x-axis) and depth (y-axis)
- Some liquefied (red), some did not (blue)
- Many lines could separate them, but SVM finds the one with the **widest gap**
- The gap is called the **margin**, and the points touching it are **support vectors**

```
         N-value →
    ┌─────────────────────┐
  D │  ●  ●              │
  e │    ●  ●    │       │  ● = Liquefiable
  p │      ●   ← margin →│  ○ = Non-liquefiable
  t │         │     ○  ○ │
  h │            ○  ○  ○ │
    └─────────────────────┘
              ↑ hyperplane
```

**Why maximize the margin?** A wider margin means the classifier is more confident and generalizes better to new data.

In [ ]:
# ─────────────────────────────────────────────────────
# IMPORTS AND SETUP
# ─────────────────────────────────────────────────────

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)

np.random.seed(44)
print("Libraries loaded successfully.")
print("Today: SVM for soil liquefaction classification.")

---

## Part 2: Generate Synthetic Liquefaction Data

We create 100 soil samples with realistic geotechnical properties. The liquefaction label is based on a simplified criterion inspired by the Seed-Idriss method:
- Low N-value + shallow depth + high water table = likely to liquefy
- High N-value + deeper + low water table = safe

In [ ]:
# ─────────────────────────────────────────────────────
# GENERATE SYNTHETIC LIQUEFACTION DATA (100 samples)
# ─────────────────────────────────────────────────────

n_samples = 100

# Features
N_value = np.random.uniform(3, 40, n_samples)          # SPT blow count
depth = np.random.uniform(1.0, 15.0, n_samples)        # Depth (m)
fines_content = np.random.uniform(2, 50, n_samples)     # Fines content (%)
water_table = np.random.uniform(0.5, 8.0, n_samples)   # Water table depth (m)

# Simplified liquefaction criterion:
# CSR-like score (higher = more likely to liquefy)
# CRR-like resistance (higher = more resistant)
csr_proxy = 0.65 * (depth / (depth + 2)) * (1 / (water_table + 0.5))
crr_proxy = 0.02 * N_value + 0.005 * fines_content + 0.01 * depth

# Add noise and classify
noise = np.random.normal(0, 0.03, n_samples)
liquefaction = ((csr_proxy - crr_proxy + noise) > 0).astype(int)

# Build DataFrame
df = pd.DataFrame({
    'N_value': np.round(N_value, 1),
    'depth_m': np.round(depth, 1),
    'fines_pct': np.round(fines_content, 1),
    'water_table_m': np.round(water_table, 1),
    'liquefaction': liquefaction
})

# Save to CSV
df.to_csv('liquefaction_data.csv', index=False)

print(f"Dataset: {df.shape[0]} samples, {df.shape[1]} columns")
print(f"\nClass distribution:")
print(f"  Non-liquefiable (0): {(df['liquefaction']==0).sum()}")
print(f"  Liquefiable (1):     {(df['liquefaction']==1).sum()}")
print(f"\nFirst 5 rows:")
df.head()

In [ ]:
# ─────────────────────────────────────────────────────
# VISUALIZE THE DATA — N-value vs Depth colored by class
# ─────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

colors = ['steelblue' if l == 0 else 'crimson' for l in df['liquefaction']]

# Plot 1: N-value vs Depth
axes[0].scatter(df['N_value'], df['depth_m'], c=colors, alpha=0.7,
                edgecolors='k', linewidth=0.5)
axes[0].set_xlabel('N-value (SPT blow count)')
axes[0].set_ylabel('Depth (m)')
axes[0].set_title('Liquefaction Data: N-value vs Depth')
axes[0].invert_yaxis()

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='steelblue', label='Non-liquefiable (0)'),
                   Patch(facecolor='crimson', label='Liquefiable (1)')]
axes[0].legend(handles=legend_elements, loc='lower right')

# Plot 2: N-value vs Water Table
axes[1].scatter(df['N_value'], df['water_table_m'], c=colors, alpha=0.7,
                edgecolors='k', linewidth=0.5)
axes[1].set_xlabel('N-value (SPT blow count)')
axes[1].set_ylabel('Water Table Depth (m)')
axes[1].set_title('Liquefaction Data: N-value vs Water Table')
axes[1].legend(handles=legend_elements, loc='lower right')

plt.tight_layout()
plt.show()
print("Key insight: Liquefiable soils tend to have lower N-values and shallower water tables.")

---

## Part 3: Data Preparation — Scaling Matters for SVM!

SVM is **distance-based** — it computes distances between points to find the margin. If features have very different scales (N-value: 3–40 vs depth: 1–15), the larger-scale feature dominates.

**Always scale features before SVM.** `StandardScaler` transforms each feature to mean=0, std=1.

In [ ]:
# ─────────────────────────────────────────────────────
# PREPARE DATA — Split and Scale
# ─────────────────────────────────────────────────────

X = df[['N_value', 'depth_m', 'fines_pct', 'water_table_m']]
y = df['liquefaction']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=44, stratify=y
)

# Scale features — fit on train, transform both
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

print(f"Training set: {X_train_sc.shape[0]} samples")
print(f"Test set:     {X_test_sc.shape[0]} samples")
print(f"\nBefore scaling (train means): {X_train.mean().values.round(1)}")
print(f"After scaling (train means):  {X_train_sc.mean(axis=0).round(4)}")
print(f"After scaling (train stds):   {X_train_sc.std(axis=0).round(4)}")
print("\nRule: Always scale for SVM. The model measures distances!")

---

## Part 4: Linear SVM — Finding the Hyperplane

A **linear SVM** fits a straight line (2D) or flat plane (higher-D) to separate the classes. It works well when classes are **linearly separable** — you can draw a straight line between them.

The `C` parameter controls the tradeoff:
- **Large C:** Strict — tries to classify every training point correctly (may overfit)
- **Small C:** Tolerant — allows some misclassifications for a wider margin (more generalizable)

In [ ]:
# ─────────────────────────────────────────────────────
# LINEAR SVM
# ─────────────────────────────────────────────────────

svm_linear = SVC(kernel='linear', C=1.0, random_state=44)
svm_linear.fit(X_train_sc, y_train)

# Predict
y_pred_linear = svm_linear.predict(X_test_sc)

print("Linear SVM Results:")
print(f"  Train accuracy: {svm_linear.score(X_train_sc, y_train):.3f}")
print(f"  Test accuracy:  {accuracy_score(y_test, y_pred_linear):.3f}")
print(f"  Support vectors: {svm_linear.n_support_}")
print(f"  Total support vectors: {sum(svm_linear.n_support_)}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred_linear,
                            target_names=['Non-Liq', 'Liquefiable']))

---

## Part 5: The Kernel Trick — Going Non-Linear

What if a straight line cannot separate the classes? The **kernel trick** maps data into a higher-dimensional space where a linear separator exists.

Common kernels:

| Kernel | When to Use | Key Parameter |
|--------|------------|---------------|
| `linear` | Linearly separable data | C |
| `rbf` (Radial Basis Function) | Most common default; curved boundaries | C, gamma |
| `poly` | Polynomial boundaries | C, degree |

**RBF kernel** creates a "bubble" of influence around each support vector. `gamma` controls the bubble size:
- **Large gamma:** Small bubbles = complex, wiggly boundary (can overfit)
- **Small gamma:** Large bubbles = smooth boundary (may underfit)

In [ ]:
# ─────────────────────────────────────────────────────
# COMPARE KERNELS: Linear, RBF, Polynomial
# ─────────────────────────────────────────────────────

kernels = {
    'Linear': SVC(kernel='linear', C=1.0, random_state=44),
    'RBF':    SVC(kernel='rbf', C=1.0, gamma='scale', random_state=44),
    'Poly (d=3)': SVC(kernel='poly', C=1.0, degree=3, random_state=44),
}

print(f"{'Kernel':<14} {'Train Acc':>10} {'Test Acc':>10} {'Support Vectors':>16}")
print("─" * 54)

results = {}
for name, model in kernels.items():
    model.fit(X_train_sc, y_train)
    train_acc = model.score(X_train_sc, y_train)
    test_acc = model.score(X_test_sc, y_test)
    n_sv = sum(model.n_support_)
    results[name] = {'train': train_acc, 'test': test_acc, 'sv': n_sv}
    print(f"{name:<14} {train_acc:>10.3f} {test_acc:>10.3f} {n_sv:>16}")

print("\nThe RBF kernel is the most common choice — it handles non-linear boundaries well.")

---

## Part 6: Visualize Decision Boundaries in 2D

To see what each kernel does, we train SVMs using only 2 features (N-value and water_table) and plot the decision boundary.

In [ ]:
# ─────────────────────────────────────────────────────
# DECISION BOUNDARY VISUALIZATION (2D)
# ─────────────────────────────────────────────────────

# Use only 2 features for visualization
X_2d = df[['N_value', 'water_table_m']].values
y_2d = df['liquefaction'].values

scaler_2d = StandardScaler()
X_2d_sc = scaler_2d.fit_transform(X_2d)

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

kernel_configs = [
    ('linear', 'Linear Kernel', {}),
    ('rbf', 'RBF Kernel', {'gamma': 'scale'}),
    ('poly', 'Polynomial (d=3)', {'degree': 3}),
]

for ax, (kern, title, params) in zip(axes, kernel_configs):
    clf = SVC(kernel=kern, C=1.0, random_state=44, **params)
    clf.fit(X_2d_sc, y_2d)
    
    # Create mesh grid for boundary
    x_min, x_max = X_2d_sc[:, 0].min() - 0.5, X_2d_sc[:, 0].max() + 0.5
    y_min, y_max = X_2d_sc[:, 1].min() - 0.5, X_2d_sc[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                         np.linspace(y_min, y_max, 200))
    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='RdBu_r')
    ax.contour(xx, yy, Z, colors='k', linewidths=0.5)
    
    for label, color, marker in [(0, 'steelblue', 'o'), (1, 'crimson', 's')]:
        mask = y_2d == label
        ax.scatter(X_2d_sc[mask, 0], X_2d_sc[mask, 1],
                   c=color, marker=marker, edgecolors='k',
                   linewidth=0.5, alpha=0.7, s=40)
    
    # Highlight support vectors
    ax.scatter(clf.support_vectors_[:, 0], clf.support_vectors_[:, 1],
               s=120, facecolors='none', edgecolors='gold', linewidths=2,
               label=f'Support Vectors ({len(clf.support_vectors_)})')
    
    acc = clf.score(X_2d_sc, y_2d)
    ax.set_title(f'{title}\nAccuracy: {acc:.3f}')
    ax.set_xlabel('N-value (scaled)')
    ax.set_ylabel('Water Table (scaled)')
    ax.legend(fontsize=8, loc='lower left')

plt.suptitle('SVM Decision Boundaries — Liquefaction Classification', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()
print("Gold circles = support vectors (the critical points defining the boundary).")
print("Notice how RBF creates a curved boundary, while linear is a straight line.")

---

## Part 7: Effect of C — Margin vs Accuracy Tradeoff

The **C parameter** is the most important tuning knob:

| C value | Margin | Training errors | Risk |
|---------|--------|----------------|------|
| Small (0.01) | Wide | Allows many | Underfitting |
| Medium (1.0) | Balanced | Some allowed | Good default |
| Large (100) | Narrow | Almost none | Overfitting |

In [ ]:
# ─────────────────────────────────────────────────────
# EFFECT OF C PARAMETER
# ─────────────────────────────────────────────────────

C_values = [0.01, 0.1, 1.0, 10, 100]

fig, axes = plt.subplots(1, 5, figsize=(22, 4))

for ax, C in zip(axes, C_values):
    clf = SVC(kernel='rbf', C=C, gamma='scale', random_state=44)
    clf.fit(X_2d_sc, y_2d)
    
    x_min, x_max = X_2d_sc[:, 0].min() - 0.5, X_2d_sc[:, 0].max() + 0.5
    y_min, y_max = X_2d_sc[:, 1].min() - 0.5, X_2d_sc[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                         np.linspace(y_min, y_max, 200))
    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='RdBu_r')
    for label, color, marker in [(0, 'steelblue', 'o'), (1, 'crimson', 's')]:
        mask = y_2d == label
        ax.scatter(X_2d_sc[mask, 0], X_2d_sc[mask, 1],
                   c=color, marker=marker, edgecolors='k', linewidth=0.5, s=30)
    
    n_sv = sum(clf.n_support_)
    acc = clf.score(X_2d_sc, y_2d)
    ax.set_title(f'C = {C}\nAcc: {acc:.3f} | SV: {n_sv}')
    ax.set_xlabel('N-value (sc.)')
    if C == 0.01:
        ax.set_ylabel('Water Table (sc.)')

plt.suptitle('Effect of C on Decision Boundary (RBF Kernel)', fontsize=13, y=1.05)
plt.tight_layout()
plt.show()
print("Small C = wide margin, simple boundary (may underfit).")
print("Large C = narrow margin, complex boundary (may overfit).")

---

## Part 8: Effect of Gamma — Influence Radius

For the RBF kernel, `gamma` controls how far each training point's influence reaches:

| Gamma | Influence | Boundary | Risk |
|-------|-----------|----------|------|
| Small (0.01) | Each point affects far away | Very smooth | Underfitting |
| Medium ('scale') | Balanced | Moderate curves | Good default |
| Large (10) | Each point only affects neighbors | Very wiggly | Overfitting |

In [ ]:
# ─────────────────────────────────────────────────────
# EFFECT OF GAMMA PARAMETER
# ─────────────────────────────────────────────────────

gamma_values = [0.01, 0.1, 1.0, 5.0, 20.0]

fig, axes = plt.subplots(1, 5, figsize=(22, 4))

for ax, g in zip(axes, gamma_values):
    clf = SVC(kernel='rbf', C=1.0, gamma=g, random_state=44)
    clf.fit(X_2d_sc, y_2d)
    
    x_min, x_max = X_2d_sc[:, 0].min() - 0.5, X_2d_sc[:, 0].max() + 0.5
    y_min, y_max = X_2d_sc[:, 1].min() - 0.5, X_2d_sc[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                         np.linspace(y_min, y_max, 200))
    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='RdBu_r')
    for label, color, marker in [(0, 'steelblue', 'o'), (1, 'crimson', 's')]:
        mask = y_2d == label
        ax.scatter(X_2d_sc[mask, 0], X_2d_sc[mask, 1],
                   c=color, marker=marker, edgecolors='k', linewidth=0.5, s=30)
    
    n_sv = sum(clf.n_support_)
    acc = clf.score(X_2d_sc, y_2d)
    ax.set_title(f'gamma = {g}\nAcc: {acc:.3f} | SV: {n_sv}')
    ax.set_xlabel('N-value (sc.)')
    if g == 0.01:
        ax.set_ylabel('Water Table (sc.)')

plt.suptitle('Effect of Gamma on Decision Boundary (RBF Kernel, C=1.0)', fontsize=13, y=1.05)
plt.tight_layout()
plt.show()
print("Small gamma = smooth boundary (underfitting risk).")
print("Large gamma = wiggly boundary that memorizes training points (overfitting risk).")

---

## Part 9: Hyperparameter Tuning with GridSearchCV

Instead of guessing C and gamma, we systematically search for the best combination using cross-validation.

In [ ]:
# ─────────────────────────────────────────────────────
# GRID SEARCH — Find optimal C and gamma
# ─────────────────────────────────────────────────────

param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': [0.01, 0.1, 1, 10],
    'kernel': ['rbf']
}

grid_search = GridSearchCV(
    SVC(random_state=44),
    param_grid,
    cv=5,
    scoring='accuracy',
    return_train_score=True,
    n_jobs=-1
)

grid_search.fit(X_train_sc, y_train)

print("Grid Search Results:")
print(f"  Best parameters: {grid_search.best_params_}")
print(f"  Best CV accuracy: {grid_search.best_score_:.3f}")
print(f"  Test accuracy:    {grid_search.score(X_test_sc, y_test):.3f}")

# Show top 5 combinations
results_df = pd.DataFrame(grid_search.cv_results_)
top5 = results_df.nsmallest(5, 'rank_test_score')[[
    'param_C', 'param_gamma', 'mean_test_score', 'std_test_score', 'mean_train_score'
]]
top5.columns = ['C', 'gamma', 'Mean CV Score', 'Std', 'Mean Train Score']
print("\nTop 5 parameter combinations:")
print(top5.to_string(index=False))

In [ ]:
# ─────────────────────────────────────────────────────
# VISUALIZE GRID SEARCH — Heatmap of C vs gamma
# ─────────────────────────────────────────────────────

scores = grid_search.cv_results_['mean_test_score'].reshape(4, 4)
C_vals = [0.1, 1, 10, 100]
gamma_vals = [0.01, 0.1, 1, 10]

fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(scores, cmap='YlOrRd', aspect='auto')

ax.set_xticks(range(4))
ax.set_yticks(range(4))
ax.set_xticklabels(gamma_vals)
ax.set_yticklabels(C_vals)
ax.set_xlabel('Gamma')
ax.set_ylabel('C')
ax.set_title('Grid Search: Mean CV Accuracy')

for i in range(4):
    for j in range(4):
        ax.text(j, i, f'{scores[i, j]:.3f}',
                ha='center', va='center', fontsize=10,
                color='white' if scores[i, j] > 0.8 else 'black')

plt.colorbar(im, label='CV Accuracy')
plt.tight_layout()
plt.show()
print("Brighter cells = higher accuracy. Best combination is the sweet spot.")

---

## Part 10: Final Evaluation with Best Model

In [ ]:
# ─────────────────────────────────────────────────────
# FINAL MODEL EVALUATION
# ─────────────────────────────────────────────────────

best_model = grid_search.best_estimator_
y_pred_best = best_model.predict(X_test_sc)

print("Final Model — Best SVM (RBF):")
print(f"  Parameters: C={best_model.C}, gamma={best_model.gamma}")
print(f"  Support vectors per class: {best_model.n_support_}")
print(f"  Total support vectors: {sum(best_model.n_support_)} / {len(y_train)} training samples")
print(f"\n  Test Accuracy: {accuracy_score(y_test, y_pred_best):.3f}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred_best,
                            target_names=['Non-Liquefiable', 'Liquefiable']))

# Confusion Matrix
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_estimator(
    best_model, X_test_sc, y_test,
    display_labels=['Non-Liq', 'Liquefiable'],
    cmap='Blues', ax=ax
)
ax.set_title('Confusion Matrix — Best SVM')
plt.tight_layout()
plt.show()

---

## Part 11: Engineering Interpretation

In [ ]:
# ─────────────────────────────────────────────────────
# ENGINEERING INTERPRETATION — Support Vector Analysis
# ─────────────────────────────────────────────────────

sv_indices = best_model.support_
sv_labels = y_train.iloc[sv_indices]

# Get original (unscaled) support vector values
sv_original = scaler.inverse_transform(X_train_sc[sv_indices])
sv_df = pd.DataFrame(sv_original, columns=['N_value', 'depth_m', 'fines_pct', 'water_table_m'])
sv_df['class'] = sv_labels.values

print("Support Vector Statistics (original scale):")
print("\nNon-liquefiable support vectors (boundary of safe zone):")
safe_sv = sv_df[sv_df['class'] == 0]
if len(safe_sv) > 0:
    print(safe_sv[['N_value', 'depth_m', 'fines_pct', 'water_table_m']].describe().round(1))

print("\nLiquefiable support vectors (boundary of danger zone):")
danger_sv = sv_df[sv_df['class'] == 1]
if len(danger_sv) > 0:
    print(danger_sv[['N_value', 'depth_m', 'fines_pct', 'water_table_m']].describe().round(1))

print("\nInterpretation:")
print("  Support vectors are the borderline cases — the soil samples closest to")
print("  the liquefaction/non-liquefaction boundary. These are the critical samples")
print("  that an engineer should investigate most carefully.")

In [ ]:
# ─────────────────────────────────────────────────────
# PREDICT NEW SOIL SAMPLES
# ─────────────────────────────────────────────────────

new_samples = pd.DataFrame({
    'N_value': [5, 15, 25, 8, 35],
    'depth_m': [2.0, 5.0, 8.0, 3.0, 12.0],
    'fines_pct': [10, 20, 5, 40, 8],
    'water_table_m': [1.0, 3.0, 5.0, 1.5, 6.0]
})

new_scaled = scaler.transform(new_samples)
predictions = best_model.predict(new_scaled)

print("New Sample Predictions:")
print(f"{'#':>3} {'N':>5} {'Depth':>6} {'Fines':>6} {'WT':>5} {'Prediction':>14}")
print("─" * 42)
for i, (_, row) in enumerate(new_samples.iterrows()):
    label = 'LIQUEFIABLE' if predictions[i] == 1 else 'Safe'
    print(f"{i+1:3d} {row['N_value']:5.0f} {row['depth_m']:5.1f}m "
          f"{row['fines_pct']:5.0f}% {row['water_table_m']:4.1f}m {label:>14}")

print("\nSamples with low N-value and shallow water table are flagged as liquefiable.")

---

## Practice Exercises

### Exercise 1: Kernel Comparison
Train an SVM with `kernel='sigmoid'` and compare its test accuracy with the other kernels. Plot the decision boundary. Does it work well for this data?

### Exercise 2: Feature Pair Boundaries
Create decision boundary plots for different feature pairs: (N_value, depth_m), (fines_pct, water_table_m). Which pair gives the best separation?

### Exercise 3: Imbalanced Data
Modify the data generation to create 80% non-liquefiable and 20% liquefiable samples. Train an SVM with `class_weight='balanced'` and compare with a regular SVM. Which handles the imbalance better?

In [ ]:
# EXERCISE 1: Sigmoid kernel
# svm_sig = SVC(kernel='sigmoid', C=1.0, random_state=44)
# svm_sig.fit(X_train_sc, y_train)
# print(f"Sigmoid test accuracy: {svm_sig.score(X_test_sc, y_test):.3f}")

In [ ]:
# EXERCISE 2: Different feature pairs
# Try: X_pair = df[['N_value', 'depth_m']].values
# Then: X_pair = df[['fines_pct', 'water_table_m']].values

In [ ]:
# EXERCISE 3: Class imbalance
# svm_balanced = SVC(kernel='rbf', C=1.0, class_weight='balanced', random_state=44)

---

## Key Takeaways

1. **SVM finds the maximum-margin boundary** between classes. Support vectors are the critical points that define this boundary — in geotechnical terms, they are the borderline soil samples.

2. **Always scale features before SVM.** Since SVM uses distances, unscaled features (N-value: 3–40 vs depth: 1–15) would give unfair weight to larger ranges.

3. **The kernel trick** lets SVM handle non-linear boundaries. RBF is the best default — it creates curved decision surfaces that capture complex liquefaction patterns.

4. **C and gamma are the key tuning parameters.** C controls margin width (strict vs tolerant), gamma controls boundary smoothness. Use GridSearchCV to find the best combination.

5. **SVM works well for binary classification** with moderate-sized datasets — exactly the kind of yes/no engineering decisions (liquefiable vs safe, stable vs unstable).

---

## How This Connects to ML

| Today's Concept | ML Application |
|----------------|----------------|
| SVM hyperplane | Decision boundary in feature space |
| Support vectors | Most informative training examples |
| Kernel trick | Non-linear classification via feature mapping |
| C parameter | Bias-variance tradeoff control |
| Gamma parameter | Model complexity control |
| Feature scaling | Essential preprocessing for distance-based models |
| GridSearchCV | Systematic hyperparameter optimization |
| Confusion matrix | Model evaluation for classification |

---

*Day 044 of 100 — SVM draws the optimal line between safe and unsafe soils!*